# `semantic.v_universe` — view

Thin view over Gold. No logic beyond shaping.

A view is its own definition, so there is no load step and no etl task to pair with this one.

In [0]:
-- THE UNIVERSE. Every trust that was ever in scope, including the ones with no prices.
-- This is the survivorship evidence and the honesty of the study in one table: a pipeline
-- that only loaded what came back would not know the erased trusts ever existed.
CREATE OR REPLACE VIEW `index-vs-trust-pipeline`.semantic.v_universe
COMMENT 'All 118 trusts and 3 index trackers, with why each is or is not in the results'
AS
SELECT d.ticker,
       d.trust_name,
       d.entity_type,
       d.management_group,
       d.manager_structure,
       d.aic_sector,
       d.status,
       d.data_status,
       d.price_source,
       d.months_available,
       d.first_month,
       d.last_month,
       d.source_url,
       CASE WHEN d.data_status = 'no-data'  THEN 'Yahoo returns nothing for it'
            WHEN d.data_status = 'excluded' THEN 'price history could not be trusted'
            WHEN d.data_status = 'stub'     THEN 'under 36 months of history'
            ELSE 'in the results'
       END                                            AS why,
       CASE WHEN d.data_status = 'usable' THEN true ELSE false END AS counts_in_the_beat_rate
FROM `index-vs-trust-pipeline`.gold.dim_ticker d
WHERE d.is_current;

## Verification

Expected: the view resolves and returns rows. Counts are in 
`
specs/04_semantic/dashboard.md
`
.

In [0]:
SELECT COUNT(*) AS rows
FROM `index-vs-trust-pipeline`.semantic.v_universe;